In [ ]:
# 10 — Export the incident history as a raw JSON feed (simulated source system)
# The synthetic history is perfectly clean, so this export simulates what a real legacy export looks like:
# ~2 % fixable noise (odd casing / whitespace) and ~1.5 % defects the silver step must catch:
#   duplicated records, empty descriptions, invalid urgency, negative distances, unparsable dates, ratings of 7.
# Defects are chosen deterministically from a hash of the incident id, so every export is identical.
# Runs once: if the export exists it is skipped (force=true rebuilds raw, bronze, silver and quarantine).

In [ ]:
import os, sys, time
sys.path.insert(0, os.getcwd())          # pipeline_lib.py lives next to this notebook
import pipeline_lib as pl
from pyspark.sql import functions as F

dbutils.widgets.text("run_id", "manual")
RUN_ID = dbutils.widgets.get("run_id")
pl.ensure_tables(spark)

dbutils.widgets.dropdown("force", "false", ["false", "true"])
FORCE  = dbutils.widgets.get("force") == "true"
MARKER = f"{pl.RAW_DIR}/_EXPORT_DONE"

def exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

In [ ]:
# Full rebuild: remove the export and everything derived from it (the live stream re-reads CDF from scratch)
if FORCE:
    for path in [pl.RAW_DIR, f"{pl.CHECKPOINT_DIR}/bronze", f"{pl.CHECKPOINT_DIR}/silver", f"{pl.CHECKPOINT_DIR}/live"]:
        dbutils.fs.rm(path, recurse=True)
    for table in [pl.T_BRONZE, pl.T_SILVER, pl.T_QUARANTINE]:
        spark.sql(f"DROP TABLE IF EXISTS {table}")
    pl.ensure_tables(spark)
    print("Reset raw export, bronze, silver, quarantine and their checkpoints")

if exists(MARKER):
    print(f"Raw export already present at {pl.RAW_DIR} — nothing to do (use force=true to rebuild)")
    dbutils.notebook.exit("skipped")

In [ ]:
started = time.time()
src = spark.table(pl.T_SOURCE)
h = F.abs(F.xxhash64("id")) % 1000          # deterministic bucket 0..999 per incident
ts = lambda c: F.date_format(c, "yyyy-MM-dd'T'HH:mm:ss")

raw = src.select(
    "id", "reported_by_user_id", "handyman_user_id",
    F.when(h.between(20, 24), F.when(h % 2 == 0, F.lit(None)).otherwise(F.lit("  ")))
     .otherwise(F.col("description")).alias("description"),
    "incident_type",
    F.when(h.between(25, 29), F.lit("URGENT!!!"))
     .when(h < 20, F.concat(F.lit("  "), F.upper("urgency"), F.lit(" ")))       # fixable noise
     .otherwise(F.col("urgency")).alias("urgency"),
    "recommended_handyman_ids", "agent_reasoning",
    F.when(h.between(30, 32), -F.col("distance_km")).otherwise(F.col("distance_km")).alias("distance_km"),
    "travel_time_minutes",
    F.when(h.between(0, 9), F.upper("status")).otherwise(F.col("status")).alias("status"),  # fixable noise
    F.when(h.between(33, 35) & (F.col("status") == "completed"), F.lit(7)).otherwise(F.col("rating")).alias("rating"),
    "feedback",
    F.when(h.between(36, 38), F.lit("31/02/2026 25:61")).otherwise(ts("created_at")).alias("created_at"),
    ts("assigned_at").alias("assigned_at"), ts("completed_at").alias("completed_at"), ts("updated_at").alias("updated_at"),
)
duplicates = raw.where(h.between(39, 43))   # ~0.5 % of records exported twice
export = raw.unionByName(duplicates)

export.repartition(64).write.mode("overwrite").json(pl.RAW_DIR)
dbutils.fs.put(MARKER, "done", overwrite=True)

n_src, n_out = src.count(), spark.read.json(pl.RAW_DIR).count()
pl.log_step(spark, RUN_ID, "10_export_raw", started, rows_in=n_src, rows_out=n_out,
            checks={"source_has_1m_rows": n_src >= 1_000_000, "export_includes_duplicates": n_out > n_src})